In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 加载model,tokenizer
custom_model_name = "fine-tuned-models/nllb-200-distilled-600M/zh2ko_1101"

model = AutoModelForSeq2SeqLM.from_pretrained(custom_model_name)
tokenizer = AutoTokenizer.from_pretrained(custom_model_name)

In [2]:
from transformers import pipeline

translator = pipeline("translation", model=model, tokenizer=tokenizer,
                      src_lang=tokenizer.src_lang,
                      tgt_lang=tokenizer.tgt_lang,
                      device="cuda:0", max_length=400, return_tensors=True)

In [3]:
import pandas as pd

df = pd.read_excel("all_files_merged_zh-CN_ko_valid_tagged.xlsx")

source = df["zh-CN"].to_list()  # 待翻译的句子
references = df["ko"].to_list()  # 标准的翻译

In [4]:
from tqdm.notebook import tqdm

# 进行翻译并提取结果
translated = [translator(item)[0] for item in tqdm(source)]
translated_token_ids = [item['translation_token_ids'].tolist() for item in translated]
translated_text = tokenizer.batch_decode(translated_token_ids, skip_special_tokens=False)

  0%|          | 0/15502 [00:00<?, ?it/s]

D:\LongtuKoreaTranslationModel\venv\lib\site-packages\transformers\pipelines\base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Your input_length: 841 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 429 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 554 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 662 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 507 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_le

In [5]:
df_compare = pd.DataFrame({
    "source": source,
    "references": references,
    "candidates": translated_text,
})
df_compare

,source,references,candidates
0,2天22<start>小时<middle>시간<end>22<start>分钟<middle...,2일 22<start>시간<end> 22<start>분<end>,</s>kor_Hang 2일 22<start> 시간<end> 22<start> 분<...
1,"{阿登丘陵的亚岱尔,找亚岱尔<start>交谈<middle>대화<end>,序章}","{아르덴 산맥의 아데어,아데어와 <start>대화<end>하기,프롤로그}","</s>kor_Hang {아르덴 산맥의 아데어,아데어와 <start> 대화<end>..."
2,<start>神秘<middle>신비<end>宝<start>箱<middle>상자<end>,<start>신비<end><start>상자<end>,</s>kor_Hang<start> 신비<end> 한 보물<start> 상자<end...
3,骑战装备熔炉<start>等级达到<middle>레벨 달성<end>4级,기마전투 장비 용로 4<start>레벨 달성<end>,</s>kor_Hang 기마전투 장비 용로 4<start> 레벨 달성<end></s>
4,<start>菜鸟<middle>신입<end>玉镯,<start>신입<end> 옥팔찌,</s>kor_Hang<start> 신입<end> 옥팔찌</s>
...,...,...,...
15497,受内攻伤害削减<code_id=1>1.5%<code_id=0>（内攻伤害包括：气功伤害、内力）,받는 내공피해 <code_id=1>1.5%<code_id=0> 감소(내공피해는 기공...,</s>kor_Hang 받는 내공피해 <code_id=1> 1.5%<code_id=...
15498,<code_id=1>双生雀<code_id=0>：谢谢浣熊大师教导，我记住了。,<code_id=1>참새부부<code_id=0>\n라쿤사부님의 가르침 감사합니다.,</s>kor_Hang<code_id=1> 참새부부<code_id=0> \n라쿤 사...
15499,"{魔力之书（精通闪避Ⅱ）,使用后学会通用技能【精通闪避Ⅱ】,<start>制作<middle...","{지식의 서 (회피 마스터리 Ⅱ),사용 후, 공통의 스킬 【회피 마스터리 Ⅱ】를 습...","</s>kor_Hang {지식의 서 (회피 마스터리 II),사용 후, 공통의 스킬 ..."
15500,正在<start>收集<middle>수집<end>{0},현재 {0} <start>수집<end>,</s>kor_Hang {0} <start> 수집<end> 중</s>


In [6]:
file_name = "translation_result_of_{0}".format(custom_model_name.replace("/", "_"))
df_compare.to_excel("{0}.xlsx".format(file_name), index=False)
df_compare.to_csv("{0}.csv".format(file_name), index=False)